# Lab 01: Scaled dot-product attention, from matmul + softmax

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hokwangchoi/hc-model-study/blob/master/labs/notebooks/lab01_attention.ipynb) · lab page: [`labs/lab01.html`](../lab01.html) · file: [`labs/lab01_attention.py`](../lab01_attention.py)

**Time:** 45-60 min

## Goal

Build the operation at the heart of every model in this repo: scaled
dot-product attention with a causal mask, and then multi-head attention
(split the width into heads, attend per head, merge back). You use only
matmul, exp and reshapes; no torch attention function.

## The maths, step by step

```text
Notation: B batch, h heads, N_q queries, N_k keys, d_k key/query width,
d_v value width, d = h * d_k model width.

  Inputs   Q [B, h, N_q, d_k]    K [B, h, N_k, d_k]    V [B, h, N_k, d_v]

  1. Scores      S = Q @ K^T / sqrt(d_k)                   [B, h, N_q, N_k]
                 S[i, j] = how well query i matches key j. Dividing by
                 sqrt(d_k) keeps the variance of S near 1, so softmax does
                 not saturate.
  2. Mask        S[i, j] = -inf wherever query i must NOT see key j.
                 Causal: query i sees keys j <= i (no peeking ahead).
  3. Softmax     P = softmax(S, dim=-1)                    [B, h, N_q, N_k]
                 P[i, j] = exp(S[i, j] - m_i) / sum_j' exp(S[i, j'] - m_i),
                 m_i = max_j S[i, j] (subtracting m_i avoids overflow and
                 does not change the result). Each row sums to 1; masked
                 entries get exactly 0.
  4. Output      O = P @ V                                 [B, h, N_q, d_v]
                 Each output row is a weighted average of the value rows.

Multi-head attention on x [B, N, d] with n_heads = h, d_k = d / h:

  q = x W_q^T + b_q,  k = x W_k^T + b_k,  v = x W_v^T + b_v   [B, N, d]
  split:  [B, N, d] -> view [B, N, h, d_k] -> transpose -> [B, h, N, d_k]
  attend: O = attention(q, k, v)                              [B, h, N, d_k]
  merge:  [B, h, N, d_k] -> transpose -> [B, N, h, d_k] -> [B, N, d]
  out = O W_o^T + b_o                                         [B, N, d]

Head i owns columns i*d_k ... (i+1)*d_k - 1 of q, k and v. That is why the
split is "view as [B, N, h, d_k], then transpose", and not "view as
[B, h, N, d_k]" (which would mix tokens and heads).
```

## Read alongside

- [Transformer, step by step](../../transformer/transformer_step_by_step.html): (the step-by-step page)
- [Transformer guide #worked-example](../../transformer/transformer_guide.html#worked-example): §2 block 8: attention with real numbers
- [Transformer guide #mha](../../transformer/transformer_guide.html#mha): §2 blocks 3–9: multi-head attention, op by op
- [Transformer guide #reshape](../../transformer/transformer_guide.html#reshape): §2 block 4: head split, [B, N, d] -> [B, h, N, d_k]
- [Transformer guide #qkt](../../transformer/transformer_guide.html#qkt): §2 block 5: why Q K^T is [B, h, N, N]
- [Transformer guide #causal-mask](../../transformer/transformer_guide.html#causal-mask): §2 block 6: the causal mask

## How to work

1. Run the setup cell and the imports cell.
2. Implement the functions in order, replacing `raise NotImplementedError  # TODO`. The hints above each cell open one at a time; try Hint 1 first.
3. Run each cell after editing it, then run the check cell at the end: `check.run_lab(1, namespace=globals())` checks the functions defined in this notebook (TODO = not started, PASS = matches the reference, FAIL = read the message).

Your work stays in this notebook; it does not change `labs/lab01_attention.py` or the lab page. The reference solution is `labs/solutions/lab01_attention.py`.

In [ ]:
# Setup: make the checker (labs/check.py) importable. Run this cell first.
# In a clone of the repository (Jupyter, VS Code) it uses labs/check.py;
# elsewhere (e.g. Google Colab) it downloads check.py, which needs only torch.
import pathlib
import sys
import urllib.request


def _find_labs_dir():
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        for cand in (d, d / "labs"):
            if (cand / "check.py").is_file() and (cand / "solutions").is_dir():
                return cand
    return None


LABS_DIR = _find_labs_dir()
if LABS_DIR is None:
    LABS_DIR = pathlib.Path("hc_model_study_labs").resolve()
    LABS_DIR.mkdir(exist_ok=True)
    if not (LABS_DIR / "check.py").is_file():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/hokwangchoi/hc-model-study/master/labs/check.py", LABS_DIR / "check.py")
sys.path.insert(0, str(LABS_DIR))
import check  # noqa: E402

import torch  # noqa: E402
print("checker:", LABS_DIR / "check.py", "| torch", torch.__version__)

### Imports

In [ ]:
import torch
import torch.nn as nn

## Step 1: the causal mask

**Hints for `causal_mask`** (open one at a time):

<details><summary>Hint 1</summary>

```text
build a column of query positions and a row of key positions and
compare them; broadcasting turns [n_q, 1] vs [1, n_k] into [n_q, n_k].
```

</details>

<details><summary>Hint 2</summary>

```text
offset = n_k - n_q; allowed[i, j] = j <= i + offset.
```

</details>

<details><summary>Hint 3</summary>

```text
torch.arange(n_q, device=device)[:, None] + offset >= torch.arange(n_k, device=device)[None, :]
```

</details>

In [ ]:
def causal_mask(n_q: int, n_k: int = None, device=None) -> torch.Tensor:
    """Boolean mask of shape [n_q, n_k]; True means "query i may attend to key j".

    With n_k == n_q (the usual case) query i sees keys 0..i, so the mask is
    lower-triangular, diagonal included.

    When there are more keys than queries (n_k > n_q) the queries are the
    LAST n_q positions of the sequence; this happens during generation with a
    KV cache (lab 03). Query i then sits at absolute position i + (n_k - n_q)
    and may attend to keys j <= i + (n_k - n_q).

    Example, n_q = 2, n_k = 4 (2 cached keys + 2 new tokens):
        [[True, True, True, False],
         [True, True, True, True ]]
    """
    if n_k is None:
        n_k = n_q
    raise NotImplementedError  # TODO

## Step 2: a numerically stable softmax

**Hints for `softmax`** (open one at a time):

<details><summary>Hint 1</summary>

```text
exp(1000) overflows float32 to inf, and inf / inf = nan.
Subtracting the row max first makes the largest exponent exp(0) = 1.
```

</details>

<details><summary>Hint 2</summary>

```text
m = x.amax(dim=dim, keepdim=True); keepdim=True keeps the shape
broadcastable against x. exp(-inf - m) is exactly 0.
```

</details>

<details><summary>Hint 3</summary>

```text
e = torch.exp(x - m); return e / e.sum(dim=dim, keepdim=True)
```

</details>

In [ ]:
def softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """softmax(x)_i = exp(x_i - max(x)) / sum_j exp(x_j - max(x)) along `dim`.

    Must not overflow for large inputs (try x = [1000., 1001., 1002.]) and
    must give exactly 0 for entries equal to -inf (masked positions).
    Do not call torch.softmax / F.softmax here; build it from torch.exp.
    """
    raise NotImplementedError  # TODO

## Step 3: scaled dot-product attention

**Hints for `scaled_dot_product_attention`** (open one at a time):

<details><summary>Hint 1</summary>

```text
four lines: scores, mask, softmax, weighted sum. Use your softmax().
```

</details>

<details><summary>Hint 2</summary>

```text
scores = q @ k.transpose(-2, -1) / q.shape[-1] ** 0.5
transpose(-2, -1) swaps the last two dims, whatever the leading dims are.
```

</details>

<details><summary>Hint 3</summary>

```text
scores = scores.masked_fill(~mask, float("-inf")) when mask is not None;
weights = softmax(scores, dim=-1); out = weights @ v
```

</details>

In [ ]:
def scaled_dot_product_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                                 mask: torch.Tensor = None):
    """Attention from matmul + softmax.

    Args:
        q:    [..., N_q, d_k]   (leading dims, e.g. [B, h], are batch dims)
        k:    [..., N_k, d_k]
        v:    [..., N_k, d_v]
        mask: optional bool tensor broadcastable to [..., N_q, N_k];
              True = may attend (same convention as causal_mask()).

    Returns:
        out:     [..., N_q, d_v]
        weights: [..., N_q, N_k]   the softmax probabilities (rows sum to 1)
    """
    raise NotImplementedError  # TODO

## Step 4: split and merge heads

**Hints for `split_heads`** (open one at a time):

<details><summary>Hint 1</summary>

```text
first split the LAST dim into (h, d_k), then move h in front of N.
```

</details>

<details><summary>Hint 2</summary>

```text
x.view(B, N, h, d_k) gives [B, N, h, d_k]; which two dims must swap?
```

</details>

<details><summary>Hint 3</summary>

```text
x.view(B, N, n_heads, d // n_heads).transpose(1, 2)
```

</details>

In [ ]:
def split_heads(x: torch.Tensor, n_heads: int) -> torch.Tensor:
    """[B, N, d] -> [B, h, N, d_k] with h = n_heads and d_k = d // h.

    Head i must hold the columns x[:, :, i*d_k:(i+1)*d_k].
    """
    raise NotImplementedError  # TODO

**Hints for `merge_heads`** (open one at a time):

<details><summary>Hint 1</summary>

```text
undo split_heads in reverse order: swap h and N back, then flatten.
```

</details>

<details><summary>Hint 2</summary>

```text
after transpose the tensor is not contiguous, so .view() fails;
use .reshape() (or .contiguous().view()).
```

</details>

<details><summary>Hint 3</summary>

```text
x.transpose(1, 2).reshape(B, N, h * d_k)
```

</details>

In [ ]:
def merge_heads(x: torch.Tensor) -> torch.Tensor:
    """[B, h, N, d_k] -> [B, N, h * d_k], the exact inverse of split_heads."""
    raise NotImplementedError  # TODO

## Step 5: multi-head attention module

**Hints for `MultiHeadAttention.forward`** (open one at a time):

<details><summary>Hint 1</summary>

```text
project -> split_heads -> scaled_dot_product_attention -> merge_heads -> out_proj.
```

</details>

<details><summary>Hint 2</summary>

```text
the mask is [N, N]; it broadcasts over [B, h, N, N] automatically.
Create it on x.device so the code also runs on a GPU.
```

</details>

<details><summary>Hint 3</summary>

```text
q = split_heads(self.q_proj(x), self.n_heads)  (same for k, v)
mask = causal_mask(N, device=x.device) if causal else None
o, w = scaled_dot_product_attention(q, k, v, mask)
return self.out_proj(merge_heads(o)), w
```

</details>

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention with separate q/k/v/out projections.

    The layers are created for you. The check copies these weights into
    torch.nn.MultiheadAttention and compares the outputs.
    """

    def __init__(self, d_model: int, n_heads: int, bias: bool = True):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=bias)
        self.k_proj = nn.Linear(d_model, d_model, bias=bias)
        self.v_proj = nn.Linear(d_model, d_model, bias=bias)
        self.out_proj = nn.Linear(d_model, d_model, bias=bias)

    def forward(self, x: torch.Tensor, causal: bool = False):
        """
        Args:
            x: [B, N, d_model]
            causal: if True, apply causal_mask(N)
        Returns:
            out:     [B, N, d_model]
            weights: [B, h, N, N]  per-head attention probabilities
        """
        raise NotImplementedError  # TODO

## Check

Run this after each function you finish. Steps you have not started show TODO; finished steps show PASS. For a full traceback of a failing step: `check.run_lab(1, globals(), verbose=True)`.

In [ ]:
check.run_lab(1, namespace=globals());

## Scratch

Try your functions here: print shapes, call them on small inputs. (Functions you have not written yet raise `NotImplementedError`.)

In [ ]:
# Try your functions here (Run the cells above first).
print(causal_mask(2, 4))
x = torch.randn(2, 5)
print(softmax(x).sum(-1))           # rows sum to 1
q = k = v = torch.randn(1, 2, 3, 4)  # [B, h, N, d_k]
out, w = scaled_dot_product_attention(q, k, v, causal_mask(3))
out.shape, w.shape